In [22]:
# JobLens - Advanced CV Parser with Docling + PyMuPDF + LLM
# Dual Mode: Docling JSON Output OR LLM-Based Parsing

# ============================================================================
# INSTALLATION & SETUP
# ============================================================================

!pip install -q openai python-docx pillow sentence-transformers scikit-learn
!pip install -q PyMuPDF  # fitz for PDF processing
!pip install -q docling  # Advanced document parsing
!pip install -q textstat

print("✅ All packages installed successfully!")

# ============================================================================
# IMPORTS
# ============================================================================

import os
import json
import base64
import re
from typing import Dict, List, Optional, Tuple
from io import BytesIO
from datetime import datetime

# File processing
import fitz  # PyMuPDF
import docx
from PIL import Image

# Docling for advanced parsing
try:
    from docling.document_converter import DocumentConverter
    from docling.datamodel.base_models import InputFormat
    DOCLING_AVAILABLE = True
except ImportError:
    print("⚠️ Docling not available, using fallback methods")
    DOCLING_AVAILABLE = False

# ML
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import textstat

# API client
from openai import OpenAI

# Colab
from google.colab import files

# ============================================================================
# CONFIGURATION
# ============================================================================

PROVIDER = "openrouter"        # "groq" or "openrouter"

# ============================================================================
# API KEYS
# ============================================================================

GROQ_API_KEY = "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"
OPENROUTER_API_KEY = "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"

# ============================================================================
# CLIENT + MODELS
# ============================================================================

if  PROVIDER == "openrouter":
    print("🎯 Using OpenRouter backend")

    client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
    )

    PARSING_MODEL = "meta-llama/llama-3.3-70b-instruct"
    PARSING_FALLBACK = "meta-llama/llama-3.1-70b-instruct"
    ATS_MODEL = "meta-llama/llama-3.3-70b-instruct"
    SCORING_MODEL = "meta-llama/llama-3.3-70b-instruct"
    OCR_MODEL = "meta-llama/llama-3.2-90b-vision-preview"

elif PROVIDER == "groq":
    print("⚡ Using Groq backend")

    client = Groq(api_key=GROQ_API_KEY)

    PARSING_MODEL = "llama-3.3-70b-versatile"
    PARSING_FALLBACK = "llama-3.1-70b-versatile"
    ATS_MODEL = "llama-3.3-70b-versatile"
    SCORING_MODEL = "llama-3.3-70b-versatile"
    OCR_MODEL = "llama-3.2-90b-vision-preview"

else:
    raise ValueError("❌ Invalid PROVIDER selected")

# Load embedding model
print("🔄 Loading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print("✅ Embedding model loaded!")

# ============================================================================
# ENHANCED PDF EXTRACTION WITH PYMUPDF (FITZ)
# ============================================================================

def extract_text_with_pymupdf(file_path: str) -> Dict:
    """Extract text and metadata from PDF using PyMuPDF (fitz)."""
    try:
        doc = fitz.open(file_path)

        result = {
            "text": "",
            "pages": [],
            "metadata": doc.metadata,
            "page_count": len(doc)
        }

        full_text = []

        for page_num in range(len(doc)):
            page = doc[page_num]
            page_text = page.get_text("text")
            full_text.append(page_text)

            page_info = {
                "page_number": page_num + 1,
                "text": page_text,
                "text_length": len(page_text)
            }

            result["pages"].append(page_info)

        result["text"] = "\n".join(full_text)

        doc.close()

        print(f"✅ PyMuPDF extracted {len(result['text'])} characters from {result['page_count']} pages")

        return result

    except Exception as e:
        print(f"❌ PyMuPDF error: {e}")
        return {"text": "", "pages": [], "metadata": {}, "page_count": 0}

# ============================================================================
# DOCLING INTEGRATION
# ============================================================================

def extract_with_docling(file_path: str) -> Dict:
    """Use Docling for advanced document parsing."""
    if not DOCLING_AVAILABLE:
        print("⚠️ Docling not available")
        return None

    try:
        print("🔄 Processing with Docling...")
        converter = DocumentConverter()

        result = converter.convert(file_path)

        docling_data = {
            "text": result.document.export_to_text(),
            "markdown": result.document.export_to_markdown(),
            "structure": str(result.document)
        }

        print(f"✅ Docling extracted {len(docling_data['text'])} characters")

        return docling_data

    except Exception as e:
        print(f"⚠️ Docling processing error: {e}")
        return None

# ============================================================================
# DOCX EXTRACTION
# ============================================================================

def extract_text_from_docx(file_path: str) -> str:
    """Extract text from DOCX with tables support."""
    try:
        doc = docx.Document(file_path)

        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]

        table_text = []
        for table in doc.tables:
            for row in table.rows:
                row_text = [cell.text.strip() for cell in row.cells]
                table_text.append(' | '.join(row_text))

        full_text = '\n'.join(paragraphs + table_text)

        print(f"✅ DOCX extracted {len(full_text)} characters")

        return full_text

    except Exception as e:
        print(f"❌ DOCX error: {e}")
        return ""

# ============================================================================
# IMAGE OCR
# ============================================================================

def image_to_base64(image_path: str) -> str:
    """Convert image to base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def extract_text_from_image(image_path: str) -> str:
    """OCR using vision model."""
    base64_image = image_to_base64(image_path)

    image_type = "image/jpeg"
    if image_path.lower().endswith('.png'):
        image_type = "image/png"

    try:
        print("🔄 Running OCR with vision model...")
        response = client.chat.completions.create(
            model=OCR_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:{image_type};base64,{base64_image}"
                            }
                        },
                        {
                            "type": "text",
                            "text": "Extract all text from this CV/resume image. Preserve structure and formatting. Return only the text."
                        }
                    ]
                }
            ],
            max_tokens=3000
        )

        text = response.choices[0].message.content
        print(f"✅ OCR extracted {len(text)} characters")
        return text

    except Exception as e:
        print(f"❌ OCR error: {e}")
        print("⚠️ Trying alternative OCR model...")

        try:
            response = client.chat.completions.create(
                model="google/gemini-flash-1.5",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:{image_type};base64,{base64_image}"
                                }
                            },
                            {
                                "type": "text",
                                "text": "Extract all visible text from this image. Return only the text content."
                            }
                        ]
                    }
                ],
                max_tokens=3000
            )

            text = response.choices[0].message.content
            print(f"✅ Alternative OCR extracted {len(text)} characters")
            return text

        except Exception as e2:
            print(f"❌ Alternative OCR failed: {e2}")
            return ""

# ============================================================================
# UNIFIED FILE PROCESSOR
# ============================================================================


def extract_links_from_pdf(file_path: str) -> Dict[str, list]:
    """
    Extract all hyperlinks from PDF pages.
    Returns dict with 'linkedin', 'github', 'others'.
    """
    try:
        doc = fitz.open(file_path)
        linkedin_links = []
        github_links = []
        other_links = []

        for page in doc:
            for link in page.get_links():
                uri = link.get("uri")
                if uri:

                    if "linkedin.com" in uri.lower():
                        if uri not in linkedin_links:
                            linkedin_links.append(uri)

                    elif "github.com" in uri.lower():
                        if uri not in github_links:
                            github_links.append(uri)
                    else:
                        if uri not in other_links:
                            other_links.append(uri)

        doc.close()

        if linkedin_links or github_links:
            print(f"🔗 Extracted links: LinkedIn={len(linkedin_links)}, GitHub={len(github_links)}")

        return {
            "linkedin": linkedin_links,
            "github": github_links,
            "others": other_links
        }

    except Exception as e:
        print(f"❌ Error extracting links from PDF: {e}")
        return {"linkedin": [], "github": [], "others": []}

#



def process_file(file_path: str) -> Tuple[str, Dict]:
    """
    Process any file type and extract text + hyperlinks.
    Returns: (text_content, links_dict)
    """
    file_extension = file_path.lower().split('.')[-1]
    links_dict = {"linkedin": [], "github": [], "others": []}

    if file_extension == 'pdf':
        print("📄 Processing PDF with PyMuPDF...")
        pymupdf_result = extract_text_with_pymupdf(file_path)
        text = pymupdf_result.get("text", "")


        links_dict = extract_links_from_pdf(file_path)

        return text, links_dict

    elif file_extension in ['docx', 'doc']:
        print("📝 Processing DOCX...")
        return extract_text_from_docx(file_path), links_dict

    elif file_extension in ['jpg', 'jpeg', 'png', 'bmp', 'tiff']:
        print("🖼️ Processing Image...")
        return extract_text_from_image(file_path), links_dict

    else:
        raise ValueError(f"Unsupported file type: {file_extension}")

# ============================================================================
# DOCLING WITH LLM FOR JSON OUTPUT
# ============================================================================

def parse_cv_with_docling_llm(file_path: str) -> Dict:
    """
    Use Docling to extract text, then use LLM to convert to JSON.
    This ensures proper JSON structure while using Docling's extraction.
    """

    print("🔄 Step 1: Extracting text with Docling + PyMuPDF...")

    # Extract with PyMuPDF
    pymupdf_result = extract_text_with_pymupdf(file_path)
    text = pymupdf_result.get("text", "")

    # ====== ADD HYPERLINK EXTRACTION HERE ======
    links_dict = extract_links_from_pdf(file_path)

    if links_dict["linkedin"]:
        linkedin_text = "\n".join(links_dict["linkedin"])
        text += f"\n\n--- EXTRACTED LINKEDIN LINKS ---\n{linkedin_text}"

    if links_dict["github"]:
        github_text = "\n".join(links_dict["github"])
        text += f"\n\n--- EXTRACTED GITHUB LINKS ---\n{github_text}"

        # Also try Docling
    docling_result = None
    if DOCLING_AVAILABLE:
        docling_result = extract_with_docling(file_path)

    # Combine both sources
    combined_text = text
    if docling_result and docling_result.get("text"):
        combined_text = text + "\n\n--- DOCLING OUTPUT ---\n\n" + docling_result["text"]

    if not combined_text:
        print("❌ Failed to extract text!")
        return get_empty_cv_template_essentials()

    print(f"✅ Extracted {len(combined_text)} characters")

    # Step 2: Use LLM to parse into JSON (ESSENTIALS ONLY)
    print("🔄 Step 2: Converting to JSON with LLM (essentials only)...")

    if len(combined_text) > 10000:
        combined_text = combined_text[:10000]
        print("⚠️ Text truncated to 10k chars")

    prompt = f"""You are an expert CV parser. Extract ESSENTIAL information from this CV/resume.

Extract ONLY these essential fields:
1. Personal Info: full_name, email, phone, linkedin
2. Professional Summary
3. Skills (technical and soft skills)
4. Work Experience

IMPORTANT INSTRUCTIONS:
- LinkedIn URL: Extract the actual hyperlink (full URL), NOT just the text "LinkedIn"
- GitHub URL: Extract the actual hyperlink (full URL), NOT just the text "GitHub"
- If hyperlinks are provided in the text sections (e.g., "--- EXTRACTED LINKEDIN LINKS ---"), use those URLs

RETURN THIS EXACT JSON STRUCTURE:
{{
  "full_name": "candidate's full name",
  "email": "email address",
  "phone": "phone number",
  "linkedin": "Full LinkedIn URL (e.g., https://linkedin.com/in/username)",
  "github": "Full GitHub URL (e.g., https://github.com/username)",
  "summary": "professional summary or objective",
  "skills": {{
    "technical": ["skill1", "skill2", "skill3"],
    "soft": ["skill1", "skill2"]
  }},
  "experience": [
    {{
      "title": "job title",
      "company": "company name",
      "duration": "time period",
      "description": "job description and responsibilities"
    }}
  ]
}}

CV TEXT:
{combined_text}

IMPORTANT: Return ONLY the JSON object. No explanations, no markdown, just pure JSON."""

    try:
        response = client.chat.completions.create(
            model=PARSING_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a CV parser. Extract essential information and return only valid JSON. No explanations, no markdown fences, just pure JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=3000,
            temperature=0.1
        )

        result = response.choices[0].message.content.strip()

        print(f"📝 Received {len(result)} characters from LLM")

        # Clean JSON
        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        parsed = json.loads(result)
        print("✅ Docling + LLM parsing successful!")

        return parsed

    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
        print(f"Response preview: {result[:300] if 'result' in locals() else 'N/A'}")

        # Try fallback model
        print(f"\n⚠️ Trying fallback model: {PARSING_FALLBACK}")
        try:
            response = client.chat.completions.create(
                model=PARSING_FALLBACK,
                messages=[
                    {"role": "system", "content": "Extract CV essentials and return only JSON. No explanations."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=3000,
                temperature=0.1
            )

            result = response.choices[0].message.content.strip()

            if '```json' in result:
                result = result.split('```json')[1].split('```')[0]
            elif '```' in result:
                parts = result.split('```')
                if len(parts) >= 2:
                    result = parts[1]

            result = result.strip()

            if not result.startswith('{'):
                start = result.find('{')
                end = result.rfind('}')
                if start != -1 and end != -1:
                    result = result[start:end+1]

            parsed = json.loads(result)
            print("✅ Fallback model succeeded!")
            return parsed

        except Exception as e2:
            print(f"❌ Fallback model failed: {e2}")

    except Exception as e:
        print(f"❌ Unexpected error: {e}")

    # Return empty template
    print("⚠️ Returning empty template")
    return get_empty_cv_template_essentials()

def get_empty_cv_template_essentials() -> Dict:
    """Return empty CV template with essentials only."""
    return {
        "full_name": "",
        "email": "",
        "phone": "",
        "linkedin": "",
        "summary": "",
        "skills": {
            "technical": [],
            "soft": []
        },
        "experience": []
    }

# ============================================================================
# LLM-BASED CV PARSING (FULL DETAILS)
# ============================================================================

def parse_cv_with_llm(cv_text: str) -> Dict:
    """Parse CV using pure LLM - extracts ALL details."""

    if len(cv_text) > 10000:
        cv_text = cv_text[:10000]
        print("⚠️ Text truncated to 10k chars")

    prompt = f"""You are an expert CV parser. Extract ALL information from this CV/resume.

EXTRACTION RULES:
1. Find and extract EVERY piece of information.
2. For contact info: email, phone, LinkedIn, GitHub, portfolio, location.
3. For skills: categorize as technical, soft, languages, tools.
4. For experience: extract ALL jobs with complete details.
5. For education: extract ALL degrees with details.
6. Extract certifications, projects, achievements.
7. If ANY field is missing, use empty string "" or empty array [].

8. Volunteer experience does NOT have to be explicitly labeled as "Volunteer".
   Infer volunteer work from context such as:
   - Student activities
   - Community work
   - NGOs or non-profit organizations
   - Unpaid roles
   - Organizing events, mentoring, teaching, helping initiatives
   If the work is unpaid or community-based, include it under "volunteer".

9. If "job_title" is not explicitly stated, infer the most appropriate job title
   based on:
   - Work experience
   - Skills
   - Projects
   - Education
   Use a standard, professional job title (e.g., "Software Engineer", "Frontend Developer", "Data Analyst").

10. The "certifications" field may include:
    - Certifications
    - Online courses
    - Training programs
    - Bootcamps
    - Workshops
    Even if they are not explicitly called a "certificate".

11. Do NOT include academic degrees (Bachelor, Master, PhD) in "certifications".

12. Do NOT guess information without strong evidence in the CV.
13. Inferred fields must be reasonable and aligned with the CV content.
14. If inference is not possible, leave the field empty.
15. IMPORTANT INSTRUCTIONS:
- LinkedIn URL: Extract the actual hyperlink (full URL), NOT just the text "LinkedIn"
- GitHub URL: Extract the actual hyperlink (full URL), NOT just the text "GitHub"
- If hyperlinks are provided in the text sections (e.g., "--- EXTRACTED LINKEDIN LINKS ---"), use those URLs

RETURN THIS EXACT JSON STRUCTURE:
{{
  "full_name": "candidate's full name",
  "email": "email address",
  "phone": "phone number with country code",
  "linkedin": "Full LinkedIn URL (e.g., https://linkedin.com/in/username)",
  "github": "Full GitHub URL (e.g., https://github.com/username)",
  "job_title": "current or target job title",
  "summary": "professional summary or objective statement",

  "education": [
    {{
      "degree": "full degree name and major",
      "institution": "university/college name",
      "location": "city, country",
      "start_date": "MM/YYYY",
      "end_date": "MM/YYYY",
      "gpa": "GPA if mentioned"
    }}
  ],
  "skills": {{
    "technical": ["Python", "JavaScript", "React", "etc"],
    "soft": ["Leadership", "Communication", "etc"],
    "tools": ["Git", "Docker", "AWS", "etc"]
  }},
  "experience": [
    {{
      "title": "exact job title",
      "company": "company name",
      "location": "city, country",
      "employment_type": "Full-time/Part-time/Contract",
      "start_date": "MM/YYYY",
      "end_date": "MM/YYYY or Present",
      "duration": "calculated duration",
      "responsibilities": ["responsibility 1", "responsibility 2"],
      "achievements": ["achievement 1", "achievement 2"],
      "technologies": ["tech used 1", "tech used 2"]
    }}
  ],
  "certifications": [
    {{
      "name": "Name of the certification, course, training, or program",
      "issuer": "Issuing organization or platform (e.g., Coursera, Udemy, Google, university, company)",
      "issue_date": "MM/YYYY"
    }}
  ],
  "projects": [
    {{
      "name": "project name",
      "description": "detailed description",
      "technologies": ["tech 1", "tech 2"],
      "date": "MM/YYYY"
    }}
  ],
  "languages": [
    {{
      "language": "language name",
      "proficiency": "Native/Fluent/Professional/Basic"
    }}
  ],
  "awards": [
    {{
      "title": "award title",
      "issuer": "organization",
      "description": "award description"
    }}
  ],
  "volunteer": [
    {{
      "role": "volunteer role",
      "organization": "organization name",
      "duration": "time period",
      "description": "what you did"
    }}
  ]
}}

CV TEXT:
{cv_text}

IMPORTANT: Return ONLY the JSON object. No explanations, no markdown, just pure JSON."""

    try:
        print(f"🤖 Parsing with {PARSING_MODEL}...")

        response = client.chat.completions.create(
            model=PARSING_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a professional CV parser. You extract information and return only valid JSON. No explanations, no markdown fences, just pure JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=4000,
            temperature=0.1
        )

        result = response.choices[0].message.content.strip()

        print(f"📝 Received {len(result)} characters from LLM")

        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        parsed = json.loads(result)
        print("✅ CV parsing successful!")
        return parsed

    except Exception as e:
        print(f"❌ Error: {e}")

    print("⚠️ Returning empty template")
    return {
        "full_name": "",
        "email": "",
        "phone": "",
        "linkedin": "",
        "github": "",
        "job_title": "",
        "summary": "",
        "education": [],
        "skills": {"technical": [], "soft": [], "tools": []},
        "experience": [],
        "certifications": [],
        "projects": [],
        "languages": [],
        "awards": [],
        "volunteer": []
    }

# ============================================================================
# LLM-BASED ATS ANALYSIS
# ============================================================================

def analyze_ats_with_llm(parsed_cv: Dict, cv_text: str) -> Dict:
    """Use LLM to analyze ATS compatibility with strict scoring criteria."""

    cv_summary = json.dumps(parsed_cv, indent=2)

    prompt = f"""You are a STRICT ATS (Applicant Tracking System) analyst and senior recruiter with 15+ years of experience.

Analyze this CV with RIGOROUS STANDARDS. Most CVs score between 50-75. Only exceptional CVs score above 85.

CV DATA:
{cv_summary}

STRICT SCORING CRITERIA:

**FORMATTING (0-100):**
- 100: Perfect ATS-friendly format, no tables/images, clear sections
- 80-90: Good format with minor issues
- 60-79: Readable but has formatting problems
- 40-59: Poor formatting, ATS may fail to parse
- 0-39: Completely unreadable by ATS

**CONTENT QUALITY (0-100):**
- 100: Every bullet uses strong action verbs + quantified results (e.g., "Increased revenue by 35%")
- 80-90: Most bullets have metrics, clear impact
- 60-79: Generic descriptions, few metrics
- 40-59: Vague descriptions like "responsible for..."
- 0-39: No measurable achievements

**KEYWORD OPTIMIZATION (0-100):**
- 100: Industry keywords naturally integrated 15+ times
- 80-90: Good keyword usage (8-14 occurrences)
- 60-79: Some keywords (4-7 occurrences)
- 40-59: Very few keywords (1-3 occurrences)
- 0-39: No relevant keywords

**EXPERIENCE PRESENTATION (0-100):**
- 100: Clear progression, quantified achievements, no gaps
- 80-90: Good career story with most achievements quantified
- 60-79: Timeline clear but lacks quantification
- 40-59: Gaps or unclear progression
- 0-39: No clear experience narrative

**SKILLS RELEVANCE (0-100):**
- 100: 15+ in-demand technical skills + soft skills
- 80-90: 10-14 relevant skills
- 60-79: 5-9 skills
- 40-59: Generic or outdated skills
- 0-39: No relevant skills listed

**COMPLETENESS (0-100):**
- 100: All sections (contact, summary, experience, education, skills, certifications)
- 80-90: Missing 1 section
- 60-79: Missing 2 sections
- 40-59: Missing 3+ sections
- 0-39: Bare minimum information

**PROFESSIONALISM (0-100):**
- 100: Perfect grammar, professional email, LinkedIn, no typos
- 80-90: 1-2 minor issues
- 60-79: 3-5 issues
- 40-59: Multiple grammar/spelling errors
- 0-39: Unprofessional presentation

**CALCULATE OVERALL SCORE:**
Overall Score = Average of all 7 categories

**GRADE MAPPING:**
- A+ (95-100): Elite, top 1%
- A (90-94): Excellent
- B+ (85-89): Very Good
- B (80-84): Good
- C+ (75-79): Above Average
- C (70-74): Average
- D (60-69): Below Average
- F (<60): Needs Major Work

**CRITICAL ISSUES (must flag):**
- Missing contact information
- No quantified achievements
- Employment gaps > 6 months unexplained
- Typos or grammar errors
- Generic objective statements
- Outdated skills

**IMPORTANT CALIBRATION EXAMPLES:**

Example 1 - Score: 45/100 (Grade F):
- Missing email
- No metrics in experience ("Worked on projects")
- Skills: "Microsoft Office, Email"
- No LinkedIn

Example 2 - Score: 68/100 (Grade D):
- Contact info present
- Experience listed but no achievements
- Generic skills
- 2-3 typos

Example 3 - Score: 82/100 (Grade B):
- All sections complete
- Some quantified achievements
- Relevant skills
- Minor formatting issues

Example 4 - Score: 93/100 (Grade A):
- Perfect format
- Every role has 3+ quantified achievements
- 15+ industry keywords
- Professional LinkedIn/GitHub
- No errors

NOW ANALYZE THIS CV:
{cv_summary}

BE STRICT. BE HONEST. Most CVs need improvement.

RETURN THIS EXACT JSON:
{{
  "overall_score": <calculated_average>,
  "grade": "<letter_grade>",
  "summary_feedback": "harsh but fair 2-sentence assessment",
  "scores": {{
    "formatting": <0-100>,
    "content_quality": <0-100>,
    "keyword_optimization": <0-100>,
    "experience_presentation": <0-100>,
    "skills_relevance": <0-100>,
    "completeness": <0-100>,
    "professionalism": <0-100>
  }},
  "strengths": [
    "specific strength with evidence",
    "another strength with evidence"
  ],
  "weaknesses": [
    "critical weakness with impact",
    "another weakness with impact"
  ],
  "critical_issues": [
    "issue 1 (if any)",
    "issue 2 (if any)"
  ],
  "improvement_suggestions": [
    {{
      "category": "category",
      "priority": "Critical/High/Medium",
      "suggestion": "actionable suggestion",
      "impact": "expected score improvement (+X points)"
    }}
  ],
  "missing_elements": ["element1", "element2"],
  "keywords_analysis": {{
    "strong_keywords": ["found1", "found2"],
    "missing_keywords": ["missing1", "missing2"],
    "suggested_keywords": ["add1", "add2"]
  }},
  "recruiter_perspective": "paragraph explaining hiring decision",
  "next_steps": ["action1", "action2"]
}}

Return ONLY JSON. NO markdown. Be brutally honest."""

    try:
        print(f"🤖 ATS analysis with {ATS_MODEL}...")

        response = client.chat.completions.create(
            model=ATS_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a STRICT ATS analyst. Most CVs score 50-75. Only exceptional CVs score 85+. Be honest and critical. Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=3000,
            temperature=0.2  # Lower temperature for more consistent scoring
        )

        result = response.choices[0].message.content.strip()

        # Clean JSON
        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        ats_result = json.loads(result)

        # Validation: Ensure score is realistic
        overall_score = ats_result.get('overall_score', 70)
        if overall_score > 95:
            print("⚠️ Score suspiciously high, recalibrating...")
            scores = ats_result.get('scores', {})
            avg = sum(scores.values()) / len(scores) if scores else 70
            ats_result['overall_score'] = round(avg)

        print(f"✅ ATS analysis complete! Score: {ats_result['overall_score']}/100")

        return ats_result

    except Exception as e:
        print(f"⚠️ ATS analysis error: {e}")

        # Fallback with more realistic defaults
        return {
            "overall_score": 65,
            "grade": "D+",
            "summary_feedback": "CV needs significant improvements in formatting and content quality.",
            "scores": {
                "formatting": 60,
                "content_quality": 55,
                "keyword_optimization": 50,
                "experience_presentation": 70,
                "skills_relevance": 65,
                "completeness": 75,
                "professionalism": 70
            },
            "strengths": ["CV structure is present"],
            "weaknesses": ["Lacks quantified achievements", "Missing keywords"],
            "critical_issues": ["No measurable results in experience section"],
            "improvement_suggestions": [
                {
                    "category": "Content",
                    "priority": "Critical",
                    "suggestion": "Add metrics to every achievement (e.g., 'Increased X by Y%')",
                    "impact": "+15 points"
                }
            ],
            "missing_elements": ["Quantified achievements"],
            "keywords_analysis": {
                "strong_keywords": [],
                "missing_keywords": ["Industry-specific terms"],
                "suggested_keywords": ["Add role-specific keywords"]
            },
            "recruiter_perspective": "CV shows potential but needs quantification and keyword optimization",
            "next_steps": ["Add metrics", "Optimize keywords", "Fix formatting"]
        }

# ============================================================================
# LLM-BASED CV IMPROVEMENTS
# ============================================================================

def generate_improvements_with_llm(parsed_cv: Dict, ats_result: Dict) -> Dict:
    """Generate personalized improvement suggestions."""

    cv_summary = json.dumps(parsed_cv, indent=2)
    ats_summary = json.dumps(ats_result, indent=2)

    prompt = f"""You are a professional CV consultant and career coach.

Based on this CV and ATS analysis, provide specific, actionable improvements.

CV:
{cv_summary}

ATS ANALYSIS:
{ats_summary}

PROVIDE DETAILED RECOMMENDATIONS IN JSON:
{{
  "priority_actions": [
    {{
      "action": "specific action to take",
      "reason": "why this matters",
      "how_to": "step-by-step how to implement",
      "priority": "Critical/High/Medium"
    }}
  ],
  "content_rewrites": {{
    "professional_summary": "rewritten professional summary that's ATS-friendly and compelling",
    "experience_improvements": [
      {{
        "current": "current description",
        "improved": "improved version with action verbs and metrics",
        "why_better": "explanation"
      }}
    ]
  }},
  "skills_strategy": {{
    "skills_to_add": ["skill 1", "skill 2"],
    "skills_to_emphasize": ["skill 1", "skill 2"],
    "skills_to_remove": ["skill 1", "skill 2"],
    "how_to_demonstrate": ["method 1", "method 2"]
  }},
  "formatting_checklist": [
    "formatting tip 1",
    "formatting tip 2"
  ],
  "keyword_strategy": {{
    "must_add": ["keyword 1", "keyword 2"],
    "contextual_usage": [
      {{
        "keyword": "keyword",
        "where": "section",
        "example": "how to use it"
      }}
    ]
  }},
  "achievement_framework": {{
    "suggested_achievements": [
      "achievement using STAR method",
      "achievement with metrics"
    ],
    "quantification_tips": ["tip 1", "tip 2"]
  }},
  "30_day_plan": [
    {{
      "week": 1,
      "tasks": ["task 1", "task 2"],
      "goal": "what to achieve"
    }}
  ]
}}

Return ONLY JSON:"""

    try:
        print(f"🤖 Generating improvements with {SCORING_MODEL}...")

        response = client.chat.completions.create(
            model=SCORING_MODEL,
            messages=[
                {"role": "system", "content": "You are a CV improvement expert. Provide actionable advice in JSON."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=3000,
            temperature=0.4
        )

        result = response.choices[0].message.content.strip()

        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        improvements = json.loads(result)
        print("✅ Improvements generated!")

        return improvements

    except Exception as e:
        print(f"⚠️ Improvements generation error: {e}")

        return {
            "priority_actions": [
                {
                    "action": "Review ATS feedback",
                    "reason": "Address identified issues",
                    "how_to": "Follow the suggestions provided",
                    "priority": "High"
                }
            ],
            "content_rewrites": {},
            "skills_strategy": {},
            "formatting_checklist": [],
            "keyword_strategy": {},
            "achievement_framework": {},
            "30_day_plan": []
        }

# ============================================================================
# ADVANCED JOB MATCHING
# ============================================================================

class JobMatcher:
    """AI-powered job matching with multi-criteria analysis."""

    def __init__(self):
        self.model = embedding_model

    def create_cv_embedding_text(self, parsed_cv: Dict) -> str:
        """Create rich text representation of CV."""
        parts = []

        if parsed_cv.get('job_title'):
            parts.append(f"Target Role: {parsed_cv['job_title']}")

        if parsed_cv.get('summary'):
            parts.append(parsed_cv['summary'])

        skills_data = parsed_cv.get('skills', {})
        if isinstance(skills_data, dict):
            all_skills = []
            for category, skills_list in skills_data.items():
                if isinstance(skills_list, list):
                    all_skills.extend(skills_list)
            if all_skills:
                parts.append(f"Skills: {', '.join(all_skills)}")

        for exp in parsed_cv.get('experience', []):
            exp_parts = []
            if exp.get('title'):
                exp_parts.append(exp['title'])
            if exp.get('company'):
                exp_parts.append(f"at {exp['company']}")
            if exp.get('description'):
                exp_parts.append(exp['description'])
            if exp.get('responsibilities'):
                if isinstance(exp['responsibilities'], list):
                    exp_parts.extend(exp['responsibilities'])
            if exp.get('achievements'):
                if isinstance(exp['achievements'], list):
                    exp_parts.extend(exp['achievements'])

            parts.append(' '.join(exp_parts))

        for proj in parsed_cv.get('projects', []):
            if proj.get('description'):
                parts.append(proj['description'])

        for cert in parsed_cv.get('certifications', []):
            if cert.get('name'):
                parts.append(f"Certified in {cert['name']}")

        return " ".join(parts)

    def match_with_llm(self, parsed_cv: Dict, job: Dict) -> Dict:
        """Use LLM for intelligent matching analysis."""

        cv_summary = json.dumps(parsed_cv, indent=2)
        job_summary = json.dumps(job, indent=2)

        prompt = f"""You are a recruitment matching expert.

Analyze how well this candidate matches this job position.

CANDIDATE CV:
{cv_summary}

JOB POSITION:
{job_summary}

PROVIDE DETAILED MATCH ANALYSIS IN JSON:
{{
  "match_score": 85,
  "match_level": "Excellent/Good/Fair/Poor",
  "recommendation": "Apply immediately / Strong candidate / Consider / Not recommended",
  "detailed_scores": {{
    "skills_match": 90,
    "experience_match": 85,
    "education_match": 80,
    "cultural_fit": 85,
    "career_trajectory": 88
  }},
  "matched_requirements": [
    "requirement 1",
    "requirement 2"
  ],
  "missing_requirements": [
    {{
      "requirement": "missing skill/experience",
      "importance": "Critical/High/Medium/Low",
      "can_learn": true/false,
      "time_to_acquire": "timeframe"
    }}
  ],
  "strengths_for_role": [
    "strength 1 specific to this role",
    "strength 2 specific to this role"
  ],
  "concerns": [
    "concern 1",
    "concern 2"
  ],
  "differentiators": [
    "what makes candidate stand out"
  ],
  "interview_talking_points": [
    "point to emphasize in interview"
  ],
  "salary_competitiveness": "assessment",
  "likelihood_of_offer": "High/Medium/Low",
  "application_strategy": "specific advice for this application"
}}

Return ONLY JSON:"""

        try:
            response = client.chat.completions.create(
                model=SCORING_MODEL,
                messages=[
                    {"role": "system", "content": "You are a recruitment expert. Analyze job matches and return JSON."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.3
            )

            result = response.choices[0].message.content.strip()

            if '```json' in result:
                result = result.split('```json')[1].split('```')[0]
            elif '```' in result:
                parts = result.split('```')
                if len(parts) >= 2:
                    result = parts[1]

            result = result.strip()

            if not result.startswith('{'):
                start = result.find('{')
                end = result.rfind('}')
                if start != -1 and end != -1:
                    result = result[start:end+1]

            return json.loads(result)

        except Exception as e:
            print(f"⚠️ LLM matching error for {job.get('title', 'Unknown')}: {e}")
            return None

    def match_jobs(self, parsed_cv: Dict, jobs: List[Dict]) -> List[Dict]:
        """Match CV against jobs using embeddings + LLM analysis."""

        print(f"🔍 Matching against {len(jobs)} jobs...")

        cv_text = self.create_cv_embedding_text(parsed_cv)
        cv_embedding = self.model.encode([cv_text])

        job_texts = [f"{job['title']}: {job['description']}" for job in jobs]
        job_embeddings = self.model.encode(job_texts)

        similarities = cosine_similarity(cv_embedding, job_embeddings)[0]

        results = []

        for i, job in enumerate(jobs):
            print(f"  Analyzing: {job['title']}...")

            semantic_score = float(similarities[i]) * 100

            llm_analysis = self.match_with_llm(parsed_cv, job)

            if llm_analysis:
                result = {
                    "job_title": job['title'],
                    "company": job.get('company', 'N/A'),
                    "location": job.get('location', 'N/A'),
                    "salary_range": job.get('salary_range', 'N/A'),
                    "semantic_similarity": round(semantic_score, 2),
                    **llm_analysis
                }
            else:
                result = {
                    "job_title": job['title'],
                    "company": job.get('company', 'N/A'),
                    "location": job.get('location', 'N/A'),
                    "salary_range": job.get('salary_range', 'N/A'),
                    "match_score": round(semantic_score, 2),
                    "match_level": self._get_match_level(semantic_score),
                    "recommendation": "Based on semantic similarity only"
                }

            results.append(result)

        results.sort(key=lambda x: x.get('match_score', 0), reverse=True)

        print("✅ Job matching complete!")

        return results

    def _get_match_level(self, score: float) -> str:
        """Get match level from score."""
        if score >= 85:
            return "🟢 Excellent"
        elif score >= 70:
            return "🟡 Good"
        elif score >= 55:
            return "🟠 Fair"
        else:
            return "🔴 Poor"

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main(parsing_mode: str = "llm"):
    """
    Main execution flow.

    Args:
        parsing_mode: "docling" or "llm"
            - "docling": Use Docling+PyMuPDF extraction → LLM for JSON (essentials only)
            - "llm": Use PyMuPDF → LLM for JSON (full details, default)
    """

    print("=" * 80)
    print("🎯 JOBLENS - AI-POWERED CV ANALYSIS")
    print(f"📋 PARSING MODE: {parsing_mode.upper()}")
    print("=" * 80)
    print()

    print("📤 Upload your CV (PDF, DOCX, or Image)...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded!")
        return

    file_name = list(uploaded.keys())[0]
    print(f"\n✅ File: {file_name}\n")

    # Parse CV based on mode
    print("=" * 80)
    if parsing_mode.lower() == "docling":
        print("STEP 1: DOCLING+PYMUPDF → LLM JSON (ESSENTIALS ONLY)")
        print("=" * 80)
        parsed_cv = parse_cv_with_docling_llm(file_name)
        cv_text = process_file(file_name)
    else:
        print("STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)")
        print("=" * 80)
        cv_text = process_file(file_name)

        if not cv_text:
            print("❌ Failed to extract text!")
            return

        print(f"✅ Extracted {len(cv_text)} characters")
        parsed_cv = parse_cv_with_llm(cv_text)

    print("\n📋 Parsed CV (JSON):")
    print(json.dumps(parsed_cv, indent=2, ensure_ascii=False))
    print()

    # ATS Analysis
    print("=" * 80)
    print("STEP 2: COMPREHENSIVE ATS ANALYSIS")
    print("=" * 80)
    ats_result = analyze_ats_with_llm(parsed_cv, cv_text)

    print(f"\n📊 ATS SCORE: {ats_result['overall_score']}/100 ({ats_result['grade']})")
    print(f"💬 {ats_result['summary_feedback']}\n")

    print("📈 Detailed Scores:")
    for category, score in ats_result.get('scores', {}).items():
        print(f"  • {category.replace('_', ' ').title()}: {score}/100")

    print(f"\n✅ Strengths:")
    for s in ats_result.get('strengths', [])[:5]:
        print(f"  • {s}")

    print(f"\n⚠️ Weaknesses:")
    for w in ats_result.get('weaknesses', [])[:5]:
        print(f"  • {w}")

    if ats_result.get('critical_issues'):
        print(f"\n🚨 Critical Issues:")
        for issue in ats_result['critical_issues']:
            print(f"  • {issue}")

    print(f"\n💡 Top Improvements:")
    for imp in ats_result.get('improvement_suggestions', [])[:3]:
        print(f"  • [{imp.get('priority', 'N/A')}] {imp.get('suggestion', 'N/A')}")

    print(f"\n👔 Recruiter Perspective:")
    print(f"  {ats_result.get('recruiter_perspective', 'N/A')}")
    print()

    # Generate improvements
    print("=" * 80)
    print("STEP 3: PERSONALIZED IMPROVEMENT PLAN")
    print("=" * 80)
    improvements = generate_improvements_with_llm(parsed_cv, ats_result)

    print("\n🎯 Priority Actions:")
    for action in improvements.get('priority_actions', [])[:3]:
        print(f"\n  [{action.get('priority', 'N/A')}] {action.get('action', 'N/A')}")
        print(f"  Why: {action.get('reason', 'N/A')}")
        print(f"  How: {action.get('how_to', 'N/A')}")

    if improvements.get('content_rewrites', {}).get('professional_summary'):
        print(f"\n📝 Suggested Professional Summary:")
        print(f"  {improvements['content_rewrites']['professional_summary']}")

    if improvements.get('skills_strategy', {}).get('skills_to_add'):
        print(f"\n🔧 Skills to Add:")
        for skill in improvements['skills_strategy']['skills_to_add'][:5]:
            print(f"  • {skill}")
    print()

    # Job matching
    print("=" * 80)
    print("STEP 4: INTELLIGENT JOB MATCHING")
    print("=" * 80)

    sample_jobs = [
        {
            "title": "Senior Python Developer",
            "company": "Tech Innovations Ltd",
            "location": "Cairo, Egypt",
            "salary_range": "60k-90k EGP",
            "description": "We're seeking a Senior Python Developer with 5+ years of experience in backend development. Must have strong experience with Django/FastAPI, PostgreSQL, Redis, and cloud platforms (AWS/GCP). Experience with microservices architecture and Docker/Kubernetes is essential. You'll lead a team of 3-5 developers and work on high-scale fintech applications.",
            "required_skills": ["Python", "Django", "FastAPI", "PostgreSQL", "Redis", "AWS", "Docker", "Kubernetes", "Microservices"],
            "required_years": 5
        },
        {
            "title": "Machine Learning Engineer",
            "company": "AI Solutions Egypt",
            "location": "Remote",
            "salary_range": "80k-120k USD",
            "description": "Join our AI team to build cutting-edge ML solutions. We need someone with strong experience in deep learning, NLP, and computer vision. Must have production experience deploying ML models at scale. TensorFlow, PyTorch, and MLOps experience required.",
            "required_skills": ["Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "NLP", "Computer Vision", "MLOps", "Kubernetes"],
            "required_years": 3
        },
        {
            "title": "Full Stack Developer",
            "company": "Startup Hub",
            "location": "Alexandria, Egypt",
            "salary_range": "40k-60k EGP",
            "description": "Fast-growing startup needs a full stack developer. React + Node.js stack. You'll build features from scratch and own the full development cycle. Startup experience is a plus.",
            "required_skills": ["React", "Node.js", "JavaScript", "TypeScript", "MongoDB", "REST API", "Git"],
            "required_years": 2
        }
    ]

    matcher = JobMatcher()
    matches = matcher.match_jobs(parsed_cv, sample_jobs)

    print("\n🎯 JOB MATCHES:\n")

    for i, match in enumerate(matches, 1):
        print(f"{i}. {match.get('job_title', 'N/A')} at {match.get('company', 'N/A')}")
        print(f"   📍 {match.get('location', 'N/A')} | 💰 {match.get('salary_range', 'N/A')}")

        if match.get('detailed_scores'):
            print(f"   📊 Breakdown:")
            for key, val in match['detailed_scores'].items():
                print(f"      • {key.replace('_', ' ').title()}: {val}/100")

        if match.get('matched_requirements'):
            print(f"   ✅ Matched Requirements ({len(match['matched_requirements'])}):")
            for req in match['matched_requirements'][:3]:
                print(f"      • {req}")

        if match.get('missing_requirements'):
            print(f"   ❌ Missing Requirements ({len(match['missing_requirements'])}):")
            for miss in match['missing_requirements'][:3]:
                if isinstance(miss, dict):
                    print(f"      • {miss.get('requirement', 'N/A')} ({miss.get('importance', 'N/A')})")
                else:
                    print(f"      • {miss}")

        if match.get('strengths_for_role'):
            print(f"   💪 Your Strengths:")
            for strength in match['strengths_for_role'][:2]:
                print(f"      • {strength}")

        print(f"   💡 Recommendation: {match.get('recommendation', 'N/A')}")

        if match.get('application_strategy'):
            print(f"   📋 Strategy: {match['application_strategy']}")

        print()

    print("=" * 80)
    print("✅ ANALYSIS COMPLETE!")
    print("=" * 80)

    return {
        "parsed_cv": parsed_cv,
        "ats_result": ats_result,
        "improvements": improvements,
        "job_matches": matches
    }

# ============================================================================
# INTERACTIVE RUNNER
# ============================================================================

def run_joblens():

    print("\n" + "=" * 80)
    print("🎯 JOBLENS - AI-POWERED CV ANALYSIS")
    print("=" * 80)
    print("\n📋 Choose CV processing mode:\n")
    print("1️⃣  Docling Mode - Extract basic CV information only")
    print("   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion")
    print("   └─ Best for: Fast analysis & essentials\n")

    print("2️⃣  LLM Mode - Full extraction of all CV details (Recommended)")
    print("   └─ Uses: PyMuPDF → LLM for JSON conversion")
    print("   └─ Best for: Comprehensive & detailed analysis\n")

    print("=" * 80)

    while True:
        choice = input("\n👉 Select mode (1 or 2): ").strip()

        if choice == "1":
            print("\n✅ Selected: Docling Mode (Basic extraction only)")
            parsing_mode = "docling"
            break
        elif choice == "2":
            print("\n✅ Selected: LLM Mode (Full analysis)")
            parsing_mode = "llm"
            break
        else:
            print("❌ Invalid choice! Please select 1 or 2")

    print("\n🚀 Starting...\n")

    result = main(parsing_mode=parsing_mode)

    return result

# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


✅ All packages installed successfully!
🎯 Using OpenRouter backend
🔄 Loading embedding model...
✅ Embedding model loaded!

🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving Rana-Nasser-Mohamed-Resume.pdf to Rana-Nasser-Mohamed-Resume (8).pdf

✅ File: Rana-Nasser-Mohamed-Resume (8).pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 4826 characters from 2 pages
🔗 Extracted links: LinkedIn=10, GitHub=5
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 8357 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Rana Nasser Mohamed",
  "email": "rananasser760@gmail.com",
  "phone": "+201225833467",
  "linkedin": "https://www.linkedin.com/in/rana-nasser-7b2375291",
  "github": "https://github.com/rananasser760",
  "job_title": "Data Engineer/Artificial Intelligence Engineer",
  "summary": "Enthusiastic Computer Science student at Ain Shams University with a strong interest in Artificial Intelligence (AI), Data Engineering, and Physics. Seeking opportunities to apply technical skills and knowledge to innovative projects in these fields.",
  

In [20]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 1

✅ Selected: Docling Mode (Basic extraction only)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: DOCLING

📤 Upload your CV (PDF, DOCX, or Image)...


[INFO] 2026-01-29 21:01:31,016 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-29 21:01:31,017 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-29 21:01:31,059 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-29 21:01:31,060 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth


Saving Rana-Nasser-Mohamed-Resume.pdf to Rana-Nasser-Mohamed-Resume (6).pdf

✅ File: Rana-Nasser-Mohamed-Resume (6).pdf

STEP 1: DOCLING+PYMUPDF → LLM JSON (ESSENTIALS ONLY)
🔄 Step 1: Extracting text with Docling + PyMuPDF...
✅ PyMuPDF extracted 4826 characters from 2 pages
🔗 Extracted links: LinkedIn=10, GitHub=5
🔄 Processing with Docling...


[INFO] 2026-01-29 21:01:31,283 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-29 21:01:31,284 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-29 21:01:31,288 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-29 21:01:31,289 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-29 21:01:31,375 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-29 21:01:31,376 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-29 21:01:31,563 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.pth
[INFO] 2026-01-29 21:01:31,570 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.pth


✅ Docling extracted 7094 characters
✅ Extracted 13455 characters
🔄 Step 2: Converting to JSON with LLM (essentials only)...
⚠️ Text truncated to 10k chars
📝 Received 2399 characters from LLM
✅ Docling + LLM parsing successful!
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 4826 characters from 2 pages
🔗 Extracted links: LinkedIn=10, GitHub=5

📋 Parsed CV (JSON):
{
  "full_name": "Rana Nasser Mohamed",
  "email": "rananasser760@gmail.com",
  "phone": "01225833467",
  "linkedin": "https://www.linkedin.com/in/rana-nasser-7b2375291",
  "github": "https://github.com/rananasser760",
  "summary": "Enthusiastic Computer Science student at Ain Shams University with a strong interest in Artificial Intelligence (AI), Data Engineering, and Physics. Seeking opportunities to apply technical skills and knowledge to innovative projects in these fields.",
  "skills": {
    "technical": [
      "Python",
      "C++",
      "C#",
      "Java",
      "JavaScript",
      "Scala",
      "SQL",
      "

In [15]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving AbdElrahman_ahmed_taher_cv.pdf to AbdElrahman_ahmed_taher_cv (7).pdf

✅ File: AbdElrahman_ahmed_taher_cv (7).pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 4235 characters from 2 pages
✅ Extracted 4235 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 4487 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Abdelrahman Ahmed Taher",
  "email": "abdelrahmaan.taheer@gmail.com",
  "phone": "+201020793558",
  "linkedin": "https://www.linkedin.com/in/Abdelrahman-Taher",
  "github": "https://github.com/AbdoTaher",
  "job_title": "Artificial Intelligence and Cross-Platform App Developer",
  "summary": "Final-year Computer Science student with a strong passion for Artificial Intelligence and cross-platform app development using Flutter.",
  "education": [
    {
      "degree": "Bachelor of Computer Science",
      "institution": "Ain Shams University",
      "location": "Cairo, E

In [23]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving AbdElrahman_ahmed_taher_cv.pdf to AbdElrahman_ahmed_taher_cv (8).pdf

✅ File: AbdElrahman_ahmed_taher_cv (8).pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 4235 characters from 2 pages
🔗 Extracted links: LinkedIn=1, GitHub=3
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 4566 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Abdelrahman Ahmed Taher",
  "email": "abdelrahmaan.taheer@gmail.com",
  "phone": "+2001020793558",
  "linkedin": "https://www.linkedin.com/in/abdelrahman-taher",
  "github": "https://github.com/abdotaher123",
  "job_title": "Artificial Intelligence and Cross-Platform App Developer",
  "summary": "Final-year Computer Science student at Ain Shams University with a strong passion for Artificial Intelligence and cross-platform app development using Flutter.",
  "education": [
    {
      "degree": "Bachelor of Computer Science",
      "

In [24]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving Mohamed_Amir_Resume.pdf to Mohamed_Amir_Resume (2).pdf

✅ File: Mohamed_Amir_Resume (2).pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 3490 characters from 1 pages
🔗 Extracted links: LinkedIn=1, GitHub=0
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 3949 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Mohamed Amir",
  "email": "mohamedamiirshabeldin@gmail.com",
  "phone": "+20 1019779488",
  "linkedin": "https://www.linkedin.com/in/mohamed-shehab25/",
  "github": "",
  "job_title": "Junior Data Analyst",
  "summary": "Motivated and detail-oriented Junior Data Analyst with a strong academic foundation in Scientific Computing. Proficient in Python, SQL, Excel, and Power BI, with a passion for leveraging data to drive decision-making. Seeking an opportunity to utilize analytical skills and technical expertise to contribute to organizational success.",
 